In [18]:
import pandas as pd

path="../data/Test_technique/p01/fitbit/"

In [19]:
steps=pd.read_json(path+"steps.json")
steps["date"]=pd.to_datetime(steps["dateTime"]).dt.date
steps_day=steps.groupby("date")["value"].sum().rename("steps")
steps_day.head()

date
2019-11-01    17873
2019-11-02    13118
2019-11-03    14312
2019-11-04    10970
2019-11-05    16186
Name: steps, dtype: int64

In [20]:
calories=pd.read_json(path+"calories.json")
calories["date"]=pd.to_datetime(calories["dateTime"]).dt.date
calories_day=calories.groupby("date")["value"].sum().rename("calories")
calories_day.head()

date
2019-11-01    4009.10
2019-11-02    3533.56
2019-11-03    3748.73
2019-11-04    3353.38
2019-11-05    3794.63
Name: calories, dtype: float64

In [21]:
distance=pd.read_json(path+"distance.json")
distance["date"]=pd.to_datetime(distance["dateTime"]).dt.date
distance_day=(distance.groupby("date")["value"].sum()/100000).rename("distance_km")
distance_day.head()

date
2019-11-01    14.42400
2019-11-02    10.58480
2019-11-03    11.46085
2019-11-04     8.85970
2019-11-05    13.71470
Name: distance_km, dtype: float64

In [22]:
daily=pd.concat([steps_day,calories_day,distance_day],axis=1)
daily.head()
#daily.describe()


,steps,calories,distance_km
date,,,
2019-11-01,17873,4009.10,14.42400
2019-11-02,13118,3533.56,10.58480
2019-11-03,14312,3748.73,11.46085
2019-11-04,10970,3353.38,8.85970
2019-11-05,16186,3794.63,13.71470


In [23]:
daily[["steps", "distance_km"]].corr()

,steps,distance_km
steps,1.000000,0.996186
distance_km,0.996186,1.000000


In [24]:
# une corr de 0.996 ->proche de 1 , donc distance probablement calculée a partir des steps


In [25]:
heart_rate=pd.read_json(path+"heart_rate.json")
heart_rate["bpm"]=heart_rate["value"].apply(lambda d: d["bpm"])
heart_rate["date"]=pd.to_datetime(heart_rate["dateTime"]).dt.date
hr_day=heart_rate.groupby("date")["bpm"].agg(["mean","min","max"])
hr_day.columns=["hr_mean","hr_min","hr_max"]
hr_day.head()

,hr_mean,hr_min,hr_max
date,,,
2019-11-01,66.140075,46,140
2019-11-02,62.602226,44,122
2019-11-03,64.551841,45,132
2019-11-04,63.552109,44,126
2019-11-05,66.672530,44,163


In [26]:
resting=pd.read_json(path+"resting_heart_rate.json")
resting["date"]=pd.to_datetime(resting["dateTime"]).dt.date
resting["rhr"]=resting["value"].apply(lambda d:d["value"])
resting_day=resting.set_index("date")["rhr"]
resting_day.head()


date
2019-11-01    53.741074
2019-11-02    52.881497
2019-11-03    53.222024
2019-11-04    54.311141
2019-11-05    52.259110
Name: rhr, dtype: float64

In [27]:
resting_day.describe()

count    152.000000
mean      52.276794
std        1.307238
min       49.031868
25%       51.285702
50%       52.222801
75%       53.088059
max       55.931519
Name: rhr, dtype: float64

In [28]:
daily=pd.concat([daily,hr_day,resting_day],axis=1)
daily.describe()

,steps,calories,distance_km,hr_mean,hr_min,hr_max,rhr
count,152.000000,152.000000,152.000000,152.000000,152.000000,152.000000,152.000000
mean,12622.881579,3607.612829,10.286953,64.342150,43.611842,138.901316,52.276794
std,4080.875255,507.267269,3.408396,4.316478,3.243025,18.969044,1.307238
min,4749.000000,2696.810000,3.824260,56.485411,30.000000,110.000000,49.031868
25%,9850.000000,3230.167500,7.915112,61.087193,43.000000,124.750000,51.285702
50%,12229.000000,3529.915000,9.873775,63.695021,44.000000,133.000000,52.222801
75%,15462.250000,3942.717500,12.753385,66.544900,45.000000,153.250000,53.088059
max,24846.000000,5225.820000,20.583600,76.938674,51.000000,191.000000,55.931519


In [29]:
#On remarque deux valeurs suspects , un minimum de 30bpm (habituellement 43 environ ) ,et un max à 191 au dessus du fc max (182) .Peut être une erreur de capteur , à voir.

In [30]:
daily[daily["hr_min"]<40]

,steps,calories,distance_km,hr_mean,hr_min,hr_max,rhr
date,,,,,,,
2020-02-11,10533,3559.41,8.48808,62.637774,36,122,52.532438
2020-02-12,10839,3473.77,8.75142,60.701950,35,129,53.021011
2020-02-22,15947,4015.10,12.87620,63.529489,33,131,52.196239
2020-02-27,19534,4697.96,15.68666,73.462873,30,166,54.844963
2020-03-09,7240,2952.43,5.84459,58.486822,33,111,51.321350
2020-03-11,8855,2892.88,7.17465,63.258794,36,120,51.556931
2020-03-14,9026,3207.16,8.17885,65.605379,33,166,50.210425
2020-03-17,7952,2932.23,6.42449,59.110301,38,136,50.059336
2020-03-19,8331,3178.23,6.66736,57.658792,35,127,49.031868


In [31]:
sleep = pd.read_json(path + "sleep.json")
main = sleep[sleep["mainSleep"] == True].copy()
main["date"] = pd.to_datetime(main["dateOfSleep"]).dt.date
main["date"].duplicated().sum()

np.int64(5)

In [32]:
doublons = main[main["date"].duplicated(keep=False)]
doublons[["date", "startTime", "endTime", "minutesAsleep", "type"]].sort_values("startTime")

,date,startTime,endTime,minutesAsleep,type
5,2019-11-07,2019-11-06 23:45:30,2019-11-07T05:52:00.000,312,stages
6,2019-11-07,2019-11-06 23:45:30,2019-11-07T05:52:00.000,312,stages
36,2019-12-07,2019-12-06 23:29:00,2019-12-07T07:21:00.000,424,stages
37,2019-12-07,2019-12-06 23:29:00,2019-12-07T07:21:00.000,424,stages
67,2020-01-06,2020-01-06 00:35:30,2020-01-06T06:25:00.000,318,stages
68,2020-01-06,2020-01-06 00:35:30,2020-01-06T06:25:00.000,318,stages
98,2020-02-05,2020-02-05 00:16:30,2020-02-05T05:54:30.000,300,stages
99,2020-02-05,2020-02-05 00:16:30,2020-02-05T05:54:30.000,300,stages
129,2020-03-06,2020-03-05 23:45:00,2020-03-06T06:44:00.000,369,stages
130,2020-03-06,2020-03-05 23:45:00,2020-03-06T06:44:00.000,369,stages


In [33]:
#on a des doublons , on les drops

In [34]:
main = main.drop_duplicates(subset=["date", "startTime", "endTime"])
main["date"].duplicated().sum()

np.int64(0)

In [35]:
levels = pd.json_normalize(main["levels"].apply(lambda d: d["summary"]))
levels.index = main.index
levels.columns

Index(['deep.count', 'deep.minutes', 'deep.thirtyDayAvgMinutes', 'wake.count',
       'wake.minutes', 'wake.thirtyDayAvgMinutes', 'light.count',
       'light.minutes', 'light.thirtyDayAvgMinutes', 'rem.count',
       'rem.minutes', 'rem.thirtyDayAvgMinutes'],
      dtype='str')

In [36]:
cols_phases = ["deep.minutes", "light.minutes", "rem.minutes", "wake.minutes"]
phases = levels.reindex(columns=cols_phases)
phases.columns = ["sleep_deep", "sleep_light", "sleep_rem", "sleep_wake"]

sleep_day = pd.concat([main[["date", "minutesAsleep", "efficiency"]], phases], axis=1)
sleep_day = sleep_day.rename(columns={"minutesAsleep": "sleep_minutes", "efficiency": "sleep_efficiency"})
sleep_day = sleep_day.set_index("date")
sleep_day.describe()

,sleep_minutes,sleep_efficiency,sleep_deep,sleep_light,sleep_rem,sleep_wake
count,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000
mean,355.066667,96.753333,39.106667,259.340000,56.620000,43.413333
std,47.826261,1.959101,17.321340,43.222358,24.648855,9.332908
min,190.000000,89.000000,0.000000,150.000000,6.000000,23.000000
25%,326.000000,96.000000,28.000000,233.000000,37.000000,38.000000
50%,349.000000,97.000000,38.000000,256.000000,59.000000,42.000000
75%,382.000000,98.000000,48.750000,281.750000,71.000000,50.000000
max,510.000000,100.000000,93.000000,378.000000,113.000000,69.000000


In [37]:
#On a 2 nuit manquantes , à voir

In [38]:
daily = pd.concat([daily, sleep_day], axis=1)
daily.isna().sum()

steps               0
calories            0
distance_km         0
hr_mean             0
hr_min              0
hr_max              0
rhr                 0
sleep_minutes       2
sleep_efficiency    2
sleep_deep          2
sleep_light         2
sleep_rem           2
sleep_wake          2
dtype: int64

In [39]:
daily[daily["sleep_minutes"].isna()]

,steps,calories,distance_km,hr_mean,hr_min,hr_max,rhr,sleep_minutes,sleep_efficiency,sleep_deep,sleep_light,sleep_rem,sleep_wake
date,,,,,,,,,,,,,
2019-11-01,17873,4009.10,14.42400,66.140075,46,140,53.741074,NaN,NaN,NaN,NaN,NaN,NaN
2020-03-11,8855,2892.88,7.17465,63.258794,36,120,51.556931,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
sleep_score = pd.read_csv(path + "sleep_score.csv")
sleep_score["date"] = pd.to_datetime(sleep_score["timestamp"]).dt.date
print(sleep_score["date"].duplicated().sum())
score_day = sleep_score.set_index("date")[["overall_score", "composition_score", "revitalization_score", "duration_score", "restlessness"]]
score_day.describe()

0


,overall_score,composition_score,revitalization_score,duration_score,restlessness
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,71.633333,17.266667,17.706667,36.660000,0.086812
std,5.634763,2.350687,2.997867,3.268376,0.029666
min,53.000000,12.000000,10.000000,25.000000,0.039437
25%,68.000000,15.000000,15.000000,35.000000,0.065046
50%,72.000000,17.000000,18.000000,37.000000,0.081021
75%,75.000000,19.000000,20.000000,38.750000,0.101530
max,88.000000,22.000000,23.000000,46.000000,0.199021


In [41]:
#test pour vérifier si il y a une relation 
(score_day["composition_score"] + score_day["revitalization_score"] + score_day["duration_score"] - score_day["overall_score"]).describe()

count    150.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
dtype: float64

In [42]:
#On remarque que les propriétés sont liées , on pourra garder 1 des 2 plus tard 

In [43]:
hr_zones = pd.read_json(path + "time_in_heart_rate_zones.json")
zones = pd.json_normalize(hr_zones["value"])
zones.index = pd.to_datetime(hr_zones["dateTime"]).dt.date
zones.columns = [c.replace("valuesInZones.", "") for c in zones.columns]
zones = zones.rename(columns={
    "BELOW_DEFAULT_ZONE_1": "zone_below",
    "IN_DEFAULT_ZONE_1": "zone_fatburn",
    "IN_DEFAULT_ZONE_2": "zone_cardio",
    "IN_DEFAULT_ZONE_3": "zone_peak",
})
zones.describe()

,zone_below,zone_fatburn,zone_peak,zone_cardio
count,152.000000,152.000000,152.000000,152.000000
mean,1263.927632,99.447368,3.473684,5.526316
std,107.841010,48.661255,7.947633,9.592317
min,574.000000,22.000000,0.000000,0.000000
25%,1238.250000,66.000000,0.000000,0.000000
50%,1297.000000,92.000000,0.000000,2.000000
75%,1332.500000,122.000000,1.000000,6.000000
max,1384.000000,328.000000,38.000000,56.000000


In [44]:
daily = pd.concat([daily, zones], axis=1)

In [45]:
pmsys = "../data/Test_technique/p01/pmsys/"

wellness = pd.read_csv(pmsys + "wellness.csv")
wellness["date"] = pd.to_datetime(wellness["effective_time_frame"]).dt.date
print(wellness["date"].duplicated().sum())

0


In [46]:
wellness_day = wellness.set_index("date")[
    ["fatigue", "mood", "readiness", "sleep_duration_h", "sleep_quality", "soreness", "stress"]
]
wellness_day.describe()


,fatigue,mood,readiness,sleep_duration_h,sleep_quality,soreness,stress
count,138.000000,138.000000,138.000000,138.000000,138.000000,138.000000,138.000000
mean,2.565217,2.963768,6.224638,5.500000,2.724638,2.543478,2.876812
std,0.539756,0.281015,1.414307,0.737425,0.479783,0.528316,0.329851
min,1.000000,1.000000,2.000000,4.000000,1.000000,1.000000,2.000000
25%,2.000000,3.000000,5.000000,5.000000,2.000000,2.000000,3.000000
50%,3.000000,3.000000,7.000000,5.000000,3.000000,3.000000,3.000000
75%,3.000000,3.000000,7.000000,6.000000,3.000000,3.000000,3.000000
max,4.000000,4.000000,8.000000,8.000000,4.000000,3.000000,3.000000


In [47]:
#Seulement 138 , on devra investiguer la dessus 

In [48]:
daily = pd.concat([daily, wellness_day], axis=1)
print(len(daily))
daily[daily["readiness"].isna()].index

152


Index([2019-11-11, 2019-12-11, 2019-12-12, 2019-12-18, 2019-12-19, 2019-12-20,
       2019-12-24, 2019-12-31, 2020-01-05, 2020-01-25, 2020-02-16, 2020-03-11,
       2020-03-26, 2020-03-31],
      dtype='object')

In [49]:
srpe = pd.read_csv(pmsys + "srpe.csv")
srpe["date"] = pd.to_datetime(srpe["end_date_time"]).dt.date
srpe["load"] = srpe["perceived_exertion"] * srpe["duration_min"]

srpe_day = srpe.groupby("date").agg(
    train_load=("load", "sum"),
    train_minutes=("duration_min", "sum"),
    n_sessions=("load", "size"),
)
srpe_day.describe()

,train_load,train_minutes,n_sessions
count,31.000000,31.000000,31.000000
mean,306.774194,50.322581,1.096774
std,136.244562,22.874130,0.300537
min,150.000000,30.000000,1.000000
25%,205.000000,30.000000,1.000000
50%,280.000000,40.000000,1.000000
75%,380.000000,60.000000,1.000000
max,660.000000,110.000000,2.000000


In [50]:
#31 jours d'entrainement sur 152 , très interessant pour le catégoriser comme une personne peu d'entrainement , mais avec une bonne 
#hygiène de vie avec son nombre de pas 

In [51]:
daily = pd.concat([daily, srpe_day], axis=1)
cols_train = ["train_load", "train_minutes", "n_sessions"]
daily[cols_train] = daily[cols_train].fillna(0)
print(len(daily))

152


In [52]:
import ast

injury = pd.read_csv(pmsys + "injury.csv")
injury["date"] = pd.to_datetime(injury["effective_time_frame"]).dt.date
injury["inj_dict"] = injury["injuries"].apply(ast.literal_eval)

injury["n_injuries"] = injury["inj_dict"].apply(len)
injury["injured"] = (injury["n_injuries"] > 0).astype(int)
injury["major"] = injury["inj_dict"].apply(lambda d: int("major" in d.values()))

print(len(injury), injury["date"].duplicated().sum())
injury[injury["injured"] == 1][["date", "injuries"]]

24 0


,date,injuries
11,2020-01-07,{'right_hand': 'minor'}


In [53]:
# 1 seul blessure en 5 mois , donc pas énormément de données à travailler 

In [54]:
#test avec les 2 autres personnes : 
for p in ["p03", "p05"]:
    inj = pd.read_csv(f"../data/Test_technique/{p}/pmsys/injury.csv")
    blessures = inj[inj["injuries"] != "{}"]
    print(p, "déclarations :", len(inj), "blessures :", len(blessures))
    display(blessures[["effective_time_frame", "injuries"]])

p03 déclarations : 11 blessures : 0


,effective_time_frame,injuries


p05 déclarations : 10 blessures : 10


,effective_time_frame,injuries
0,2019-11-01T06:30:35.565Z,{'left_foot': 'minor'}
1,2019-11-07T07:29:02.805Z,{'head_neck': 'minor'}
2,2020-01-30T06:38:02.691Z,{'left_foot': 'minor'}
3,2020-01-31T05:40:48.569Z,{'left_foot': 'minor'}
4,2020-02-08T04:59:19.621Z,{'left_foot': 'minor'}
5,2020-02-14T05:32:10.455Z,{'left_foot': 'minor'}
6,2020-02-29T06:26:35.754Z,{'left_foot': 'minor'}
7,2020-03-07T08:01:19.947Z,{'left_foot': 'minor'}
8,2020-03-15T06:01:04.240Z,{'left_foot': 'minor'}
9,2020-03-17T19:47:51.329Z,{'left_foot': 'minor'}


In [55]:
# On remarque qu'on a 10 blessure pour p05 et sur le pieds gauche presque toujours , donc peut être une blessure qui
# c'est aggravé 

# Aucune données réel de blessure pour p03 

In [56]:
injury_day = injury.set_index("date")[["injured", "major"]]
daily = pd.concat([daily, injury_day], axis=1)
print(len(daily))


152
